# Hackathon Dataset Cleaning and Join Analysis

Objective: turn the FDR healthcare-facility data into a cleaner, uncertainty-aware dataset for non-technical planners, medical staff, and researchers.

This notebook follows the hackathon warning from the screenshots: the facility fields are extracted claims from open web text, not verified ground truth. The cleaning step therefore keeps claim evidence, join strategy, confidence, and uncertainty-review flags.


## Screenshot-Derived Context

- Source pipeline: web crawl -> GenAI extraction -> entity resolution -> FDR dataset.
- Dataset: roughly 10k India-focused facility records with 51 facility columns.
- Required behavior: cite underlying facility text, communicate uncertainty honestly, and persist review decisions.
- Prior notes: merging records creates uncertainty; useful analysis includes distributions, confidence intervals, Boolean indicators, Bayesian-style confidence, active-learning-style uncertainty triage, and sensitivity analysis without pretending proxy labels are ground truth.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "output" / "data" / "facility_health_cleaned.csv").exists():
        ROOT = candidate
        break
DATA_DIR = ROOT / "output" / "data"
PLOT_DIR = ROOT / "output" / "plots"

facility_clean = pd.read_csv(DATA_DIR / "facility_health_cleaned.csv")
district_clean = pd.read_csv(DATA_DIR / "district_health_facility_cleaned.csv")
unmatched_pincode_districts = pd.read_csv(DATA_DIR / "district_unmatched_pincode_facility_counts.csv")
summary = json.loads((DATA_DIR / "analysis_summary.json").read_text())
clean = facility_clean  # Alias used by the facility QA cells below.
district_clean.shape, facility_clean.shape, summary["source_tables"]


## Primary Output: District-Level Dataset

District is the safer analytic grain. NFHS health indicators are district-level, India Post lets us bridge PIN codes to district/state, and facility rows are noisy web-extracted claims. The district dataset aggregates facility evidence and quality signals without pretending that each facility record is fully verified.


In [ ]:
district_cols = [
    "state_ut", "district_name", "observed_facility_rows",
    "health_need_score", "district_medical_desert_priority_score",
    "district_data_quality_score", "district_uncertainty_level",
    "source_url_rate", "needs_human_review_rate", "sample_facility_names"
]
district_clean[district_cols].head(20)


## Join Strategy

Facilities do not carry a reliable district field. The defensible join path is:

1. Extract a six-digit Indian PIN from `address_zipOrPostcode`.
2. Join the PIN to India Post.
3. Collapse India Post rows to the modal district/state for each PIN while preserving ambiguity counts.
4. Normalize state and district names.
5. Join to NFHS district health indicators on normalized state + district.
6. Use facility city/state only as a low-confidence fallback when PIN is missing.

The cleaned dataset keeps `join_strategy`, `join_confidence`, `join_match_score`, and `join_uncertainty_reason` so downstream users can filter or review risky rows.


In [ ]:
clean["join_strategy"].value_counts(dropna=False).to_frame("rows")


![Join strategy distribution](../../output/plots/join_strategy_distribution.png)


## Field Coverage and Claim Risk

High field coverage is useful but not equivalent to truth. Description, procedure, equipment, and capability text can be used as evidence snippets, but the scores/rankings should cite those fields and disclose that they are extracted claims.


![Field coverage](../../output/plots/field_coverage.png)


In [ ]:
coverage = pd.DataFrame(summary["field_coverage"]).T
coverage.sort_values("present_pct")


## Distribution Findings

The facility numeric fields are not Gaussian. They are sparse, parsed from text, and heavy-tailed. For ranking and cleaning, the pipeline uses log-scale plots, percentile ranks, and outlier flags instead of deleting high values.


![Numeric distributions](../../output/plots/numeric_distributions_log.png)


In [ ]:
pd.DataFrame(summary["distribution_profiles"]).T


## Geography and Join Uncertainty

Coordinates include off-India values and rows far from their pincode centroid. Those rows are flagged rather than silently dropped because they may reflect extraction errors, entity-resolution mistakes, or facilities with international web artifacts.


![Geo quality scatter](../../output/plots/geo_quality_scatter.png)


In [ ]:
clean["geo_quality"].value_counts(dropna=False).to_frame("rows")


## Medical Desert Proxy

Without true catchment population or verified facility supply, this is a proxy, not a definitive desert label. The score combines:

- NFHS district-level health need percentile.
- Inverse percentile of facility count observed in the joined FDR sample.

Use this to prioritize review and planning questions, not to make final policy claims.


![Medical desert proxy](../../output/plots/medical_desert_proxy_scatter.png)


In [ ]:
cols = [
    "state_ut", "district_name", "observed_facility_rows",
    "health_need_score", "district_medical_desert_priority_score",
    "district_data_quality_score", "district_uncertainty_level",
    "predominant_join_uncertainty", "sample_facility_names"
]
district_clean.sort_values("district_medical_desert_priority_score", ascending=False)[cols].head(20)


## Data Readiness

The readiness score is deliberately conservative. It rewards parseable pincode, health join, plausible coordinates, non-ambiguous PIN bridge, source URLs, claim-text coverage, and contact evidence. Rows below the threshold or with outlier flags are marked `needs_human_review`, which now means uncertainty review rather than manual verification.


![Data readiness distribution](../../output/plots/data_readiness_distribution.png)


![District data quality distribution](../../output/plots/district_data_quality_distribution.png)


In [ ]:
clean["needs_human_review"].value_counts(dropna=False).to_frame("rows")


## Active Uncertainty Queue

Without human verification labels, active learning becomes active uncertainty triage. The queue ranks facilities and districts where source enrichment, stress testing, or cautious UI treatment would most reduce decision uncertainty. The scores are not measured accuracy; they combine clinical impact, semantic missingness, contradiction risk, decision leverage, sparse-segment coverage, geocoding uncertainty, and confidence/prediction-interval width.


In [ ]:
facility_queue = pd.read_csv(DATA_DIR / "active_learning_facility_queue.csv")
district_queue = pd.read_csv(DATA_DIR / "active_learning_district_queue.csv")

facility_queue[
    [
        "active_uncertainty_rank",
        "active_uncertainty_score",
        "active_learning_action",
        "facility_name",
        "state_ut",
        "district_name",
        "proxy_trust_interval_low",
        "proxy_trust_interval_high",
        "external_validation_action",
        "pre_geocode_uncertainty_band_low_km",
        "pre_geocode_uncertainty_band_high_km",
        "capacity_estimate_interval_low",
        "capacity_estimate_interval_high",
        "doctor_count_estimate_interval_low",
        "doctor_count_estimate_interval_high",
    ]
].head(15)


## External Evidence and Geocoding Uncertainty

The parallel geocoding work addresses the garbage-in/garbage-out problem by turning Google Maps, Mappls, India Post, and registry evidence into uncertainty explanations. A geocoder result is not treated as truth by itself. It only reduces uncertainty when its `status`, `location_type`, `partial_match`, formatted address, place ID, and coordinates agree with the expected state, district, pincode, and facility name.

The active-learning idea here is value-of-information triage: spend external API calls and registry lookups on rows where location disagreement is large and the district decision would change.


![Geo external validation actions](../../output/plots/geo_external_validation_actions.png)


![Geo external validation priority scatter](../../output/plots/geo_external_priority_scatter.png)


In [ ]:
geo_candidates_path = DATA_DIR / "geo_validation_candidates.csv"
geocoder_priors_path = DATA_DIR / "geocoder_uncertainty_priors.csv"

geo_candidates = pd.read_csv(geo_candidates_path) if geo_candidates_path.exists() else pd.DataFrame()
geocoder_priors = pd.read_csv(geocoder_priors_path) if geocoder_priors_path.exists() else pd.DataFrame()

if geo_candidates.empty:
    print("No geo validation candidate file found yet. Run scripts/prepare_geo_validation_batch.py to generate it.")
else:
    display(
        geo_candidates[
            [
                "facility_name",
                "state_ut",
                "district_name",
                "geo_review_reason",
                "external_validation_action",
                "external_validation_priority_score",
                "geo_distance_km_to_pincode_centroid",
                "pre_geocode_uncertainty_band_low_km",
                "pre_geocode_uncertainty_band_high_km",
                "geocoder_expected_precision_after_success",
            ]
        ].head(15)
    )


In [ ]:
if not geocoder_priors.empty:
    display(geocoder_priors)


In [ ]:
if not geo_candidates.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    geo_candidates["external_validation_action"].value_counts().sort_values().plot(
        kind="barh", ax=axes[0], color="#3c6e71"
    )
    axes[0].set_title("External validation actions")
    axes[0].set_xlabel("Candidate rows")
    axes[0].set_ylabel("")

    by_reason = (
        geo_candidates.groupby("geo_review_reason")["external_validation_priority_score"]
        .mean()
        .sort_values()
    )
    by_reason.plot(kind="barh", ax=axes[1], color="#d98c3a")
    axes[1].set_title("Mean source-enrichment priority")
    axes[1].set_xlabel("Priority score")
    axes[1].set_ylabel("")

    plt.tight_layout()
    plt.show()


In [ ]:
if not geo_candidates.empty:
    plot_df = geo_candidates.copy()
    plot_df["distance_for_plot_km"] = pd.to_numeric(
        plot_df["geo_distance_km_to_pincode_centroid"], errors="coerce"
    ).clip(lower=0, upper=2500)
    plot_df["external_validation_priority_score"] = pd.to_numeric(
        plot_df["external_validation_priority_score"], errors="coerce"
    )

    fig, ax = plt.subplots(figsize=(11, 6))
    for reason, group in plot_df.groupby("geo_review_reason"):
        ax.scatter(
            group["distance_for_plot_km"],
            group["external_validation_priority_score"],
            label=reason,
            alpha=0.72,
            s=42,
        )
    ax.set_xscale("symlog", linthresh=10)
    ax.set_xlabel("Distance from India Post PIN centroid, km (capped at 2,500)")
    ax.set_ylabel("External validation priority score")
    ax.set_title("Which geo failures should use external API calls first?")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if not geo_candidates.empty:
    interval_df = geo_candidates.copy()
    interval_df["uncertainty_band_width_km"] = (
        pd.to_numeric(interval_df["pre_geocode_uncertainty_band_high_km"], errors="coerce")
        - pd.to_numeric(interval_df["pre_geocode_uncertainty_band_low_km"], errors="coerce")
    ).clip(lower=0)
    top_intervals = interval_df.nlargest(15, "uncertainty_band_width_km")[
        [
            "facility_name",
            "state_ut",
            "district_name",
            "geo_review_reason",
            "external_validation_action",
            "uncertainty_band_width_km",
            "geocoder_acceptance_rule",
        ]
    ]
    display(top_intervals)


In [ ]:
district_queue[
    [
        "active_uncertainty_rank",
        "active_uncertainty_score",
        "active_learning_action",
        "state_ut",
        "district_name",
        "care_gap_score",
        "trust_gap_score",
        "needs_human_review_rate_ci_low",
        "needs_human_review_rate_ci_high",
        "critical_supply_gap_rate_ci_low",
        "critical_supply_gap_rate_ci_high",
        "trustworthy_supply_rate_ci_low",
        "trustworthy_supply_rate_ci_high",
    ]
].head(15)


## Exported Artifacts

- `output/data/facility_health_cleaned.csv`
- `output/data/district_health_facility_cleaned.csv` (primary cleaned dataset)
- `output/data/district_unmatched_pincode_facility_counts.csv`
- `output/data/active_learning_facility_queue.csv`
- `output/data/active_learning_district_queue.csv`
- `output/data/geo_validation_candidates.csv`
- `output/data/geocoder_uncertainty_priors.csv`
- `output/data/facility_health_cleaned_data_dictionary.csv`
- `output/data/district_health_facility_cleaned_data_dictionary.csv`
- `output/data/analysis_summary.json`
- `output/plots/*.png`


In [ ]:
district_path = DATA_DIR / "district_health_facility_cleaned.csv"
facility_audit_path = DATA_DIR / "facility_health_cleaned.csv"
dictionary_path = DATA_DIR / "district_health_facility_cleaned_data_dictionary.csv"
district_path, facility_audit_path, dictionary_path
